# Train EfficientDet-D4

Notebook training workflow for the prepared CXRaide/VinBig 1024x1024 object-detection dataset. It uses `effdet`'s `tf_efficientdet_d4` model with pretrained weights and writes checkpoints to `notebooks/outputs/models/efficientdet_d4`.

## 1. Environment

Run the install cell once per environment. Restart the kernel after installing packages if Jupyter asks you to.

In [ ]:
# Uncomment if your environment does not already have these packages.
# %pip install -U torch torchvision effdet timm pandas pillow tqdm pyyaml mlflow

In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import mlflow
import pandas as pd
import torch
from effdet import create_model
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as F
from tqdm.auto import tqdm

## 2. Configuration

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "ML":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "notebooks" / "data"
ANNOTATION_CSV = DATA_DIR / "processed" / "02_vinbig_transformed_bounding_boxes_1024.csv"
IMAGE_DIR = DATA_DIR / "raw" / "RAW_VINBIG" / "images"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "models" / "efficientdet_d4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 1024
BATCH_SIZE = 2
EPOCHS = 10
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.15
NUM_WORKERS = 0  # Keep 0 on Windows/Jupyter unless you know multiprocessing is stable.
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

## 3. Load Prepared Annotations

In [ ]:
df = pd.read_csv(ANNOTATION_CSV)
df = df.rename(columns={"class_name": "label_name"})
required_columns = {"image_id", "label_name", "target_class_id", "x_min", "y_min", "x_max", "y_max"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df.dropna(subset=list(required_columns)).copy()
for col in ["x_min", "y_min", "x_max", "y_max"]:
    df[col] = df[col].astype(float).clip(0, IMAGE_SIZE)
df = df[(df["x_max"] > df["x_min"]) & (df["y_max"] > df["y_min"])]
df["target_class_id"] = df["target_class_id"].astype(int)

label_map = (
    df[["target_class_id", "label_name"]]
    .drop_duplicates()
    .sort_values("target_class_id")
    .set_index("target_class_id")["label_name"]
    .to_dict()
)
NUM_CLASSES = len(label_map)

print(f"Annotations: {len(df):,}")
print(f"Images with boxes: {df['image_id'].nunique():,}")
print(f"Classes: {NUM_CLASSES}")
label_map

In [ ]:
def image_path_for(image_id: str) -> Path | None:
    for suffix in (".png", ".jpg", ".jpeg"):
        candidate = IMAGE_DIR / f"{image_id}{suffix}"
        if candidate.exists():
            return candidate
    return None


image_paths = {image_id: image_path_for(str(image_id)) for image_id in df["image_id"].unique()}
available_ids = [image_id for image_id, path in image_paths.items() if path is not None]
missing_count = len(image_paths) - len(available_ids)
if missing_count:
    print(f"Skipping {missing_count:,} image ids without local image files.")

df = df[df["image_id"].isin(available_ids)].copy()
if df.empty:
    raise FileNotFoundError(f"No training images found under {IMAGE_DIR}")

image_ids = sorted(df["image_id"].unique())
random.Random(SEED).shuffle(image_ids)
val_count = max(1, int(len(image_ids) * VAL_FRACTION))
val_ids = set(image_ids[:val_count])
train_ids = set(image_ids[val_count:])

train_df = df[df["image_id"].isin(train_ids)].copy()
val_df = df[df["image_id"].isin(val_ids)].copy()

print(f"Train images: {train_df['image_id'].nunique():,} / boxes: {len(train_df):,}")
print(f"Val images:   {val_df['image_id'].nunique():,} / boxes: {len(val_df):,}")

## 4. Dataset and Dataloaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class CXRDetectionDataset(Dataset):
    def __init__(self, annotations: pd.DataFrame, image_paths: dict[str, Path], image_size: int) -> None:
        self.annotations = annotations
        self.image_paths = image_paths
        self.image_size = image_size
        self.image_ids = sorted(annotations["image_id"].unique())
        self.grouped = {image_id: group for image_id, group in annotations.groupby("image_id")}

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, idx: int):
        image_id = self.image_ids[idx]
        image = Image.open(self.image_paths[image_id]).convert("RGB")
        original_w, original_h = image.size

        boxes = self.grouped[image_id][["x_min", "y_min", "x_max", "y_max"]].to_numpy(dtype="float32")
        if (original_w, original_h) != (self.image_size, self.image_size):
            scale_x = self.image_size / original_w
            scale_y = self.image_size / original_h
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y
            image = image.resize((self.image_size, self.image_size), Image.BILINEAR)

        labels = self.grouped[image_id]["target_class_id"].to_numpy(dtype="int64")
        image_tensor = F.to_tensor(image)
        image_tensor = F.normalize(image_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)

        target = {
            "bbox": torch.as_tensor(boxes, dtype=torch.float32),
            "cls": torch.as_tensor(labels, dtype=torch.int64),
            "img_size": torch.tensor([self.image_size, self.image_size], dtype=torch.float32),
            "img_scale": torch.tensor(1.0, dtype=torch.float32),
            "image_id": image_id,
        }
        return image_tensor, target


def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images), {
        "bbox": [target["bbox"] for target in targets],
        "cls": [target["cls"] for target in targets],
        "img_size": torch.stack([target["img_size"] for target in targets]),
        "img_scale": torch.stack([target["img_scale"] for target in targets]),
        "image_id": [target["image_id"] for target in targets],
    }

In [ ]:
train_dataset = CXRDetectionDataset(train_df, image_paths, IMAGE_SIZE)
val_dataset = CXRDetectionDataset(val_df, image_paths, IMAGE_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)

sample_images, sample_targets = next(iter(train_loader))
sample_images.shape, sample_targets["bbox"][0].shape, sample_targets["cls"][0]

## 5. EfficientDet-D4 Model

In [ ]:
model = create_model(
    "tf_efficientdet_d4",
    bench_task="train",
    num_classes=NUM_CLASSES,
    pretrained=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

## 6. Training Helpers

In [ ]:
def move_targets_to_device(targets: dict, device: torch.device) -> dict:
    return {
        "bbox": [boxes.to(device) for boxes in targets["bbox"]],
        "cls": [labels.to(device) for labels in targets["cls"]],
        "img_size": targets["img_size"].to(device),
        "img_scale": targets["img_scale"].to(device),
    }


def loss_value(output):
    if isinstance(output, dict):
        return output["loss"]
    return output


def train_one_epoch(epoch: int) -> float:
    model.train()
    total_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False)
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True)
        targets = move_targets_to_device(targets, DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            output = model(images, targets)
            loss = loss_value(output)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / max(1, len(train_loader))


@torch.no_grad()
def validate_one_epoch(epoch: int) -> float:
    model.eval()
    total_loss = 0.0
    progress = tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False)
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True)
        targets = move_targets_to_device(targets, DEVICE)
        output = model(images, targets)
        loss = loss_value(output)
        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / max(1, len(val_loader))


def save_checkpoint(path: Path, epoch: int, val_loss: float) -> None:
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_loss": val_loss,
            "label_map": label_map,
            "image_size": IMAGE_SIZE,
            "model_name": "tf_efficientdet_d4",
        },
        path,
    )

## 7. Train

In [ ]:
mlflow.set_tracking_uri((PROJECT_ROOT / "mlruns").as_uri())
mlflow.set_experiment("cxraide-efficientdet-d4")

best_val_loss = float("inf")
best_checkpoint = OUTPUT_DIR / "best_tf_efficientdet_d4.pt"
last_checkpoint = OUTPUT_DIR / "last_tf_efficientdet_d4.pt"

with mlflow.start_run(run_name="tf_efficientdet_d4_1024"):
    mlflow.log_params(
        {
            "model": "tf_efficientdet_d4",
            "image_size": IMAGE_SIZE,
            "num_classes": NUM_CLASSES,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "train_images": train_df["image_id"].nunique(),
            "val_images": val_df["image_id"].nunique(),
        }
    )

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(epoch)
        val_loss = validate_one_epoch(epoch)
        scheduler.step()

        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss}, step=epoch)
        save_checkpoint(last_checkpoint, epoch, val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(best_checkpoint, epoch, val_loss)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | best={best_val_loss:.4f}"
        )

    mlflow.log_artifact(str(best_checkpoint))
    mlflow.log_artifact(str(last_checkpoint))

print(f"Best checkpoint: {best_checkpoint}")

## 8. Inference Bench Sanity Check

Use the inference bench after training when you want quick visual/model-output checks from the saved weights.

In [ ]:
# checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
# infer_model = create_model(
#     "tf_efficientdet_d4",
#     bench_task="predict",
#     num_classes=NUM_CLASSES,
#     pretrained=False,
#     image_size=(IMAGE_SIZE, IMAGE_SIZE),
# )
# infer_model.load_state_dict(checkpoint["model_state_dict"], strict=False)
# infer_model = infer_model.to(DEVICE).eval()
# images, targets = next(iter(val_loader))
# with torch.no_grad():
#     detections = infer_model(images.to(DEVICE))
# detections.shape